# Exercise 06b — Select traffic regimes and compression capacity

Use internal criteria for the GMM and validation reconstruction error for PCA/autoencoder capacity. The test period is kept untouched until one setting per method has been frozen.

## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `06 Unsupervised learning`, so relative data paths work.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "06 Unsupervised learning"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics. PyTorch builds the neural networks used in this lesson.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn torch


## Prepared traffic data and chronological windows

Each 16-interval row is both input and reconstruction target. Scaling is fitted on training windows only; all reported errors and examples are converted back to vehicles per 15 minutes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

COLORS = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})
torch.manual_seed(42)
np.random.seed(42)

raw = pd.read_csv("Traffic_Flow_Data.csv")
times = raw.columns[1:-1]
values = raw[times].to_numpy(dtype=float)
series = values.reshape(-1)

window_length = 16
X_raw = np.stack([
    series[start:start + window_length]
    for start in range(len(series) - window_length + 1)
])
window_end_step = np.arange(window_length - 1, len(series))
window_end_day = window_end_step // 96
window_end_interval = window_end_step % 96

train_mask = window_end_day < 20
valid_mask = (window_end_day >= 20) & (window_end_day < 25)
test_mask = window_end_day >= 25

scaler = StandardScaler().fit(X_raw[train_mask])
X_scaled = scaler.transform(X_raw).astype("float32")
X_train = X_scaled[train_mask]
X_valid = X_scaled[valid_mask]
X_test = X_scaled[test_mask]

print("X_train:", X_train.shape, "target:", X_train.shape)
print("X_valid:", X_valid.shape, "target:", X_valid.shape)
print("X_test :", X_test.shape, "target:", X_test.shape)
print("There is no separate y: a reconstruction model learns X -> compressed z -> reconstructed X.")
display(pd.DataFrame({"one raw traffic window": X_raw[train_mask][0]}).T)

## Question 1 — Select the number of traffic regimes

Fit GMMs with 2–5 components to training flow observations. Use BIC to select the component count, then draw the histogram and each weighted Gaussian density in original flow units. Name the regimes by their ordered means.

In [ ]:
# TODO: fit 2-5 component GMMs on training flow and choose the minimum-BIC model.
# TODO: plot the observed density and weighted component densities in raw units.

## Question 2 — Select PCA compression capacity

Try latent dimensions 2, 4 and 8. Fit PCA on `X_train`, calculate validation reconstruction MAE in original units, and freeze the dimension with the smallest validation MAE. Also report retained variance.

In [ ]:
# TODO: compare PCA latent sizes 2, 4 and 8 using validation reconstruction MAE.
# TODO: report retained variance and freeze the best size.

## Question 3 — Select autoencoder capacity

Implement a nonlinear autoencoder and compare the same three latent sizes. Use validation error for selection and store the best weights for each candidate. This is still unsupervised: the target is the input window itself.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_size, latent_size, hidden_size=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, latent_size),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size),
        )

    def forward(self, x):
        latent = self.encoder(x)
        return self.decoder(latent)


def train_autoencoder(latent_size, train_array, valid_array, epochs=350):
    torch.manual_seed(42 + latent_size)
    model = Autoencoder(train_array.shape[1], latent_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_function = nn.MSELoss()
    train_tensor = torch.tensor(train_array)
    valid_tensor = torch.tensor(valid_array)
    best_validation_loss = float("inf")
    best_state = None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_function(model(train_tensor), train_tensor)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_loss = loss_function(model(valid_tensor), valid_tensor).item()
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = {
                name: parameter.detach().clone()
                for name, parameter in model.state_dict().items()
            }

    model.load_state_dict(best_state)
    return model

In [ ]:
# TODO: train latent sizes 2, 4 and 8 and compare validation MAE in raw units.
# TODO: freeze the best latent size and its best validation checkpoint.

## Question 4 — Refit, test once and diagnose time-of-day error

After freezing one dimension per method, refit on train + validation, evaluate the untouched test period once, and plot mean reconstruction MAE by time of day. Show one well reconstructed and one poorly reconstructed test window.

In [ ]:
# TODO: refit each selected method on train + validation and evaluate test once.
# TODO: plot reconstruction error by time of day and two representative windows.

## Question 5 — Extension: reconstruction error as an anomaly score

Rank test windows by reconstruction MAE. Plot the top five scores against their end times and inspect their raw profiles. Explain why a large reconstruction error is a screening signal, not proof of an incident.

In [ ]:
# TODO: rank test windows by reconstruction MAE and visualise the top five.
# TODO: explain why anomaly scores require external confirmation.